# 2.14 深度学习训练基础 (Deep Learning Training Basics)深度学习是大模型所有技术的基础，其核心训练机制支撑着上述所有学习范式。本节涵盖：- 反向传播- 优化器- 学习率调度- 正则化- 归一化- 梯度管理

## 1. 反向传播**目的**：高效计算神经网络中所有参数的梯度**基本原理**：使用链式法则从输出层向输入层逐层计算偏导数。

In [ ]:
import torchimport torch.nn as nntorch.manual_seed(42)model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))X = torch.randn(4, 10)y = torch.randint(0, 3, (4,))loss_fn = nn.CrossEntropyLoss()logits = model(X)loss = loss_fn(logits, y)loss.backward()print('=== Backpropagation ===')print(f'Loss: {loss.item():.4f}')print(f'\nGradient shapes:')for name, param in model.named_parameters():    if param.grad is not None:        print(f'  {name:20s}: param={param.shape}, grad_norm={param.grad.norm():.4f}')print('\nBackprop computes gradients for ALL parameters in a single forward+backward pass.')

## 2. 优化器**目的**：高效地更新模型参数

In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimtorch.manual_seed(42)model_sgd = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))model_adam = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))model_adamw = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))model_adam.load_state_dict(model_sgd.state_dict())model_adamw.load_state_dict(model_sgd.state_dict())X = torch.randn(64, 10)y = torch.randint(0, 3, (64,))opt_sgd = optim.SGD(model_sgd.parameters(), lr=0.01)opt_adam = optim.Adam(model_adam.parameters(), lr=1e-3)opt_adamw = optim.AdamW(model_adamw.parameters(), lr=1e-3, weight_decay=0.01)loss_fn = nn.CrossEntropyLoss()print('=== Optimizer Comparison ===')for epoch in range(20):    for model, optimizer, name in [(model_sgd, opt_sgd, 'SGD'), (model_adam, opt_adam, 'Adam'), (model_adamw, opt_adamw, 'AdamW')]:        loss = loss_fn(model(X), y)        optimizer.zero_grad(); loss.backward(); optimizer.step()        if (epoch+1) % 5 == 0:        losses = {}        for model, name in [(model_sgd, 'SGD'), (model_adam, 'Adam'), (model_adamw, 'AdamW')]:            with torch.no_grad():                l = loss_fn(model(X), y).item()            losses[name] = l        print(f'Epoch {epoch+1}: ' + ', '.join(f'{k}={v:.4f}' for k, v in losses.items()))print('\nKey: AdamW decouples weight decay from gradient update (standard for LLM training).')

## 3. 学习率调度 / 4. 正则化 / 5. 归一化 / 6. 梯度管理

In [ ]:
import torchimport torch.nn as nnimport mathtorch.manual_seed(42)# Learning Rate Schedulingprint('=== Learning Rate Schedules ===')n_steps = 1000warmup_steps = 100peak_lr = 1e-3def cosine_schedule(step, warmup, peak_lr, total_steps):    if step < warmup:        return peak_lr * step / warmup    progress = (step - warmup) / (total_steps - warmup)    return peak_lr * 0.5 * (1 + math.cos(math.pi * progress))def wsd_schedule(step, warmup, stable_steps, peak_lr, total_steps):    if step < warmup:        return peak_lr * step / warmup    elif step < warmup + stable_steps:        return peak_lr    else:        progress = (step - warmup - stable_steps) / (total_steps - warmup - stable_steps)        return peak_lr * 0.5 * (1 + math.cos(math.pi * progress))steps = list(range(0, n_steps, 50))cos_lrs = [cosine_schedule(s, warmup_steps, peak_lr, n_steps) for s in steps]wsd_lrs = [wsd_schedule(s, warmup_steps, 500, peak_lr, n_steps) for s in steps]print(f'{"Step":>6} {"Cosine":>10} {"WSD":>10}')for s, c, w in zip(steps, cos_lrs, wsd_lrs):    print(f'{s:>6} {c:>10.6f} {w:>10.6f}')# Normalizationprint('\n=== Normalization ===')x = torch.randn(4, 10)layer_norm = nn.LayerNorm(10)rms_norm_output = x / torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + 1e-8)print(f'LayerNorm output mean: {layer_norm(x).mean(dim=-1)}')print(f'RMSNorm output norm: {rms_norm_output.norm(dim=-1)}')print('RMSNorm is faster (no mean subtraction) and used in LLaMA, Mistral.')# Gradient Managementprint('\n=== Gradient Management ===')model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))loss = nn.CrossEntropyLoss()(model(torch.randn(4, 10)), torch.randint(0, 3, (4,)))loss.backward()grad_norm = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None]).norm()print(f'Gradient norm before clipping: {grad_norm:.4f}')max_norm = 1.0torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)grad_norm_after = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None]).norm()print(f'Gradient norm after clipping (max_norm={max_norm}): {grad_norm_after:.4f}')print('\nGradient clipping prevents training instability from gradient explosions.')